# Bronze: ClinVar variant reference (NCBI E-utilities)

Pulls germline variant records for a childhood epilepsy gene panel from the **public
NCBI ClinVar API**. No credentials, no uploads, no patient data.

| | |
| --- | --- |
| **Source** | `eutils.ncbi.nlm.nih.gov` — `esearch` then `esummary` against `db=clinvar` |
| **Writes** | `bronze_clinvar_variants`, `bronze_reference_release` |
| **Cited as** | VCV accession, plus the release date this run read |

Bronze stores what the API returned, verbatim. A ClinVar classification is only
meaningful against a stated release, so the release date is captured as data rather
than left implicit — reclassification is routine and a report issued last month may
not hold today.

In [ ]:
PANEL_NAME = "childhood_epilepsy_v1"
GENES = "SCN1A,SCN2A,KCNQ2,STXBP1,CDKL5,MECP2,PCDH19,DEPDC5"
MAX_PER_GENE = 120
PIPELINE_RUN_ID = ""

In [ ]:
import json
import time
import uuid
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, IntegerType, StringType, StructField, StructType,
)

RUN_ID = PIPELINE_RUN_ID or str(uuid.uuid4())
INGESTED_AT = datetime.now(timezone.utc).isoformat()
GENE_LIST = [g.strip() for g in GENES.split(",") if g.strip()]

print(json.dumps({"run_id": RUN_ID, "panel": PANEL_NAME, "genes": GENE_LIST}, indent=1))

In [ ]:
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
HEADERS = {"User-Agent": "fabric-genetic-consultation-demo/1.0"}


def _get(endpoint, params, attempts=4):
    """NCBI rate-limits anonymous callers; back off rather than hammering."""
    last = None
    for attempt in range(attempts):
        try:
            response = requests.get(f"{EUTILS}/{endpoint}", params=params,
                                    headers=HEADERS, timeout=90)
            if response.status_code == 200:
                return response.json()
            last = f"HTTP {response.status_code}"
        except Exception as error:
            last = str(error)[:120]
        time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"{endpoint} failed after {attempts} attempts: {last}")


def search_gene(symbol, retmax):
    payload = _get("esearch", {"db": "clinvar", "retmode": "json", "retmax": retmax,
                               "term": f"{symbol}[gene] AND single_gene[prop]"})
    return payload.get("esearchresult", {}).get("idlist", [])


def summarise(uids):
    """esummary in batches; the API rejects very long id lists."""
    records = []
    for start in range(0, len(uids), 50):
        batch = uids[start:start + 50]
        payload = _get("esummary", {"db": "clinvar", "retmode": "json",
                                    "id": ",".join(batch)})
        result = payload.get("result", {})
        for uid in result.get("uids", []):
            records.append(result[uid])
        time.sleep(0.4)
    return records

In [ ]:
raw_rows = []
coverage_rows = []

for symbol in GENE_LIST:
    try:
        uids = search_gene(symbol, MAX_PER_GENE)
        records = summarise(uids) if uids else []
        for record in records:
            raw_rows.append({
                "run_id": RUN_ID,
                "gene_symbol": symbol,
                "uid": str(record.get("uid")),
                "accession": record.get("accession"),
                "title": record.get("title"),
                "obj_type": record.get("obj_type"),
                "record_status": record.get("record_status"),
                "protein_change": record.get("protein_change"),
                "record_json": json.dumps(record),
                "ingested_at_utc": INGESTED_AT,
            })
        coverage_rows.append({
            "run_id": RUN_ID, "source_name": f"ClinVar - {symbol}",
            "table_name": "bronze_clinvar_variants", "gene_symbol": symbol,
            "status": "available" if records else "empty",
            "record_count": len(records), "note": None,
            "checked_at_utc": INGESTED_AT,
        })
        print(f"  {symbol:8} {len(records):4} records")
    except Exception as error:
        # An unreachable gene is a coverage gap, not zero variants. The distinction has
        # to survive into the report, so it is recorded rather than swallowed.
        coverage_rows.append({
            "run_id": RUN_ID, "source_name": f"ClinVar - {symbol}",
            "table_name": "bronze_clinvar_variants", "gene_symbol": symbol,
            "status": "unavailable", "record_count": 0,
            "note": str(error)[:300], "checked_at_utc": INGESTED_AT,
        })
        print(f"  {symbol:8} UNAVAILABLE - {str(error)[:90]}")

print(f"\ntotal variant records: {len(raw_rows)}")

In [ ]:
VARIANT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("gene_symbol", StringType(), True),
    StructField("uid", StringType(), True),
    StructField("accession", StringType(), True),
    StructField("title", StringType(), True),
    StructField("obj_type", StringType(), True),
    StructField("record_status", StringType(), True),
    StructField("protein_change", StringType(), True),
    StructField("record_json", StringType(), True),
    StructField("ingested_at_utc", StringType(), True),
])

COVERAGE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("gene_symbol", StringType(), True),
    StructField("status", StringType(), True),
    StructField("record_count", IntegerType(), True),
    StructField("note", StringType(), True),
    StructField("checked_at_utc", StringType(), True),
])


def write_run_scoped(dataframe, table_name):
    """Partition by run_id so many runs coexist and any query can pin one."""
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING: {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:110]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


def to_rows(records, schema):
    columns = [field.name for field in schema.fields]
    return spark.createDataFrame(
        [tuple(record.get(column) for column in columns) for record in records],
        schema=schema)


write_run_scoped(to_rows(raw_rows, VARIANT_SCHEMA), "bronze_clinvar_variants")
write_run_scoped(to_rows(coverage_rows, COVERAGE_SCHEMA), "bronze_reference_coverage")

In [ ]:
# The release stamp. Which ClinVar release a classification came from is clinical
# metadata, not provenance trivia -- assertions are revised continuously.
release_rows = [{
    "run_id": RUN_ID,
    "source": "ClinVar (NCBI)",
    "access_method": "E-utilities esearch/esummary, db=clinvar",
    "read_at_utc": INGESTED_AT,
    "panel_name": PANEL_NAME,
    "genes_requested": len(GENE_LIST),
    "genes_returned": sum(1 for row in coverage_rows if row["status"] == "available"),
    "variant_records": len(raw_rows),
}]
RELEASE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("source", StringType(), True),
    StructField("access_method", StringType(), True),
    StructField("read_at_utc", StringType(), True),
    StructField("panel_name", StringType(), True),
    StructField("genes_requested", IntegerType(), True),
    StructField("genes_returned", IntegerType(), True),
    StructField("variant_records", IntegerType(), True),
])
write_run_scoped(to_rows(release_rows, RELEASE_SCHEMA), "bronze_reference_release")

print(json.dumps(release_rows[0], indent=1))

In [ ]:
display(spark.read.table("bronze_clinvar_variants")
        .filter(F.col("run_id") == RUN_ID)
        .select("gene_symbol", "accession", "title", "protein_change")
        .limit(15))